# Sesión 7 · Diseño del proyecto

**Módulo 4 · Tu primera app de IA** · 2 sesiones · proyecto real · de idea a producto

> Llegamos al proyecto final: construiremos un **chatbot de Preguntas Frecuentes (FAQ) para atención al cliente**. El cliente escribe su duda y el asistente responde basándose en un PDF de FAQ. Hoy diseñamos la solución y montamos el primer prototipo; en la Sesión 8 lo puliremos y lo publicaremos.

**Al terminar la sesión sabrás:**
- Definir un problema real y diseñar su solución
- Entender la arquitectura **datos → modelo → interfaz**
- Leer un PDF y convertirlo en contexto para un LLM
- Crear una primera interfaz web con **Streamlit**

*Resultado de la sesión: tienes un prototipo funcional del chatbot.*

> 📎 **Necesitas el archivo `faq_horta_verde.pdf`** (te lo paso junto al notebook). Colócalo en la misma carpeta. En Google Colab, súbelo con el panel de archivos de la izquierda.

## 1. El problema: ¿qué resolvemos y para quién?

**El caso.** Una tienda online (la ficticia *Horta Verde*) recibe cada día las mismas preguntas: plazos de envío, devoluciones, formas de pago... Su equipo pierde horas respondiendo lo mismo y, fuera del horario de oficina, el cliente se queda sin respuesta.

**La solución.** Un asistente disponible 24/7 que responde esas dudas al instante, **usando como única fuente la información oficial de la empresa** (su documento de FAQ). Ni se inventa nada ni sustituye al equipo humano: lo libera de lo repetitivo.

**Por qué encaja un LLM aquí.** Las preguntas llegan en lenguaje natural y de mil formas distintas ("¿cuándo llega?", "plazo de entrega?", "para cuándo lo tengo?"). Un sistema de reglas no cubriría todas; un LLM entiende la intención y responde con naturalidad. Es justo lo que vimos en el Módulo 3.

> **Principio de diseño:** un buen primer proyecto resuelve **un problema pequeño y concreto**. Mejor un chatbot que domina las FAQ de una tienda que un "asistente que lo sabe todo".

## 2. La arquitectura: datos → modelo → interfaz

Toda app de IA, por compleja que parezca, sigue este esquema de tres bloques:

```
   DATOS                MODELO                  INTERFAZ
┌──────────┐        ┌──────────────┐        ┌──────────────┐
│ PDF de   │  ───>  │ LLM (Claude, │  ───>  │ Web de chat  │
│ las FAQ  │        │ Llama...)    │        │ (Streamlit)  │
└──────────┘        └──────────────┘        └──────────────┘
   El texto         Recibe las FAQ           El cliente
   de referencia    + la pregunta y          escribe y lee
                    redacta la respuesta     la respuesta
```

**¿Cómo le "damos" las FAQ al modelo?** Aquí tomamos una decisión de diseño importante. Como el documento de FAQ es **corto**, usamos la técnica más simple y efectiva: **meter todo el texto de las FAQ dentro del *system prompt***. El modelo recibe en cada consulta las reglas + las FAQ completas + la pregunta del cliente. A esto se le llama, de forma informal, *"context-stuffing"* (rellenar el contexto).

> Cuando el documento es enorme (cientos de páginas), meterlo entero no cabe en la ventana de contexto y se usa otra técnica llamada **RAG** (búsqueda + generación). La dejamos para los próximos pasos (Sesión 8). Para unas FAQ, *context-stuffing* es la opción correcta y sencilla.

## 3. Paso 1 · Leer el PDF de FAQ

Empezamos por los **datos**. Usamos la librería `pypdf` para extraer el texto del PDF. Ejecuta las celdas (instala primero si hace falta).

In [ ]:
!pip install pypdf anthropic openai ollama streamlit -q
print("Listo ✅")

In [ ]:
from pypdf import PdfReader

lector = PdfReader("faq_horta_verde.pdf")
texto_faq = ""
for pagina in lector.pages:
    texto_faq += pagina.extract_text() + "\n"

print("Páginas leídas:", len(lector.pages))
print("Caracteres extraídos:", len(texto_faq))
print("\n--- Primeras líneas ---")
print(texto_faq[:400])

## 4. Paso 2 · Construir el prompt

Ahora preparamos las instrucciones del asistente (el **system prompt**) aplicando lo aprendido en la Sesión 5: le damos un **rol**, unas **reglas claras** y la **fuente de información** (las FAQ). La regla más importante: que responda **solo** con lo que está en las FAQ y que, si no lo sabe, lo diga.

In [ ]:
system = f"""Eres el asistente de atención al cliente de Horta Verde, una tienda online.
Responde en español, de forma amable y breve.

Reglas:
- Usa SOLO la información de las FAQ de abajo.
- Si la respuesta no está, dilo e invita a escribir a hola@hortaverde.example. No inventes.

=== FAQ ===
{texto_faq}
=== FIN ==="""

print("System prompt preparado.")
print("Longitud (caracteres):", len(system))

## 5. Paso 3 · Preguntar al LLM

Probamos la lógica antes de envolverla en una web. Reutilizamos lo del Módulo 3. Introduce tu clave y lanza una pregunta de prueba.

> Mostramos el ejemplo con **Claude**, pero la app final permitirá elegir entre los tres proveedores (Claude, OpenRouter gratuito y Ollama local), exactamente como en la Sesión 6.

In [ ]:
import os
from getpass import getpass
os.environ["ANTHROPIC_API_KEY"] = getpass("Clave de Anthropic: ")

In [ ]:
import anthropic

def responder_faq(pregunta):
    cliente = anthropic.Anthropic()
    r = cliente.messages.create(
        model="claude-haiku-4-5",
        max_tokens=400,
        system=system,
        messages=[{"role": "user", "content": pregunta}],
    )
    return r.content[0].text

# Una pregunta que SÍ está en las FAQ
print(responder_faq("¿Cuánto cuesta el envío?"))
print("\n----\n")
# Una pregunta que NO está -> debería reconocer que no lo sabe
print(responder_faq("¿Tenéis tienda física en Madrid?"))

## 6. La interfaz: ¿qué es Streamlit?

**Streamlit** es una librería que convierte un script de Python en una **web interactiva**, sin saber nada de desarrollo web. Tú escribes Python y Streamlit pone los botones, las cajas de texto y el chat.

Hay una diferencia clave con lo que hemos hecho hasta ahora: una app de Streamlit **no se ejecuta dentro del notebook**, sino como un programa aparte. El flujo es:
1. Escribes el código en un archivo `.py`
2. Lo lanzas desde la terminal con `streamlit run mi_app.py`
3. Se abre en el navegador

Desde el notebook usaremos la orden mágica `%%writefile` para **guardar** el código en un archivo, y luego lo ejecutaremos en la terminal.

In [ ]:
%%writefile hola_streamlit.py
import streamlit as st

st.title("¡Hola, Streamlit!")
nombre = st.text_input("¿Cómo te llamas?")
if nombre:
    st.write(f"Encantado, {nombre} 👋")

La celda anterior **no muestra nada**: solo ha guardado el archivo `hola_streamlit.py`. Para verlo en marcha, abre una terminal en la misma carpeta y ejecuta:

```bash
streamlit run hola_streamlit.py
```

> En **Google Colab**, ejecutar Streamlit necesita un paso extra (un túnel). La forma más cómoda de seguir esta parte es en tu **ordenador**: instala Python, o usa el editor que prefieras, y lanza los comandos `streamlit run` en su terminal. Si trabajas en Colab, dímelo y te paso la receta del túnel.

## 7. Tu primer prototipo

Juntamos las tres piezas (leer PDF + prompt + LLM) dentro de una app de Streamlit mínima. Esta versión hace lo esencial: una caja para la pregunta, un botón y la respuesta. Sin memoria de conversación todavía: eso lo añadimos en la Sesión 8.

La siguiente celda **guarda** el prototipo en `app_v1.py`.

In [ ]:
%%writefile app_v1.py
# -*- coding: utf-8 -*-
"""Prototipo v1 del chatbot de FAQ (Sesión 7).
Versión mínima: una pregunta, una respuesta. Sin historial todavía.

Ejecutar con:  streamlit run app_v1.py
"""
import os
import streamlit as st
from pypdf import PdfReader

st.title("Asistente de FAQ (prototipo)")

# 1. Leer el PDF de FAQ
lector = PdfReader("faq_horta_verde.pdf")
texto_faq = ""
for pagina in lector.pages:
    texto_faq += pagina.extract_text() + "\n"

# 2. Pedir la clave de Claude (solo para el prototipo; luego mejoraremos esto)
clave = st.text_input("Tu clave de Anthropic", type="password")

# 3. Caja para la pregunta del cliente
pregunta = st.text_input("¿En qué puedo ayudarte?")

# 4. Al pulsar el botón, preguntamos al LLM
if st.button("Preguntar") and pregunta and clave:
    import anthropic
    cliente = anthropic.Anthropic(api_key=clave)
    system = (
        "Eres el asistente de atención al cliente de Horta Verde. "
        "Responde en español, breve y amable, usando SOLO esta información:\n\n"
        + texto_faq
    )
    respuesta = cliente.messages.create(
        model="claude-haiku-4-5",
        max_tokens=400,
        system=system,
        messages=[{"role": "user", "content": pregunta}],
    )
    st.write(respuesta.content[0].text)


Ahora ejecútalo en tu terminal:

```bash
streamlit run app_v1.py
```

Se abrirá en el navegador. Introduce tu clave, escribe *"¿cuánto tarda el envío?"* y pulsa **Preguntar**. ¡Acabas de usar tu primera app de IA! 🎉

👉 **Pruébalo con 5 preguntas distintas**, algunas que estén en las FAQ y otras que no. Anota cómo se comporta: lo mejoraremos juntos.

## 8. Resumen de la sesión

- Un buen proyecto resuelve **un problema pequeño y concreto**: aquí, las FAQ de una tienda
- Toda app de IA es **datos → modelo → interfaz**
- Para un documento corto, la técnica más simple es **meter las FAQ en el system prompt** (*context-stuffing*)
- **Streamlit** convierte un script de Python en una web; se lanza con `streamlit run`
- Ya tienes un **prototipo funcional** (`app_v1.py`)

**Para la próxima sesión:** lo convertimos en un producto: interfaz de chat con memoria, lógica más robusta, despliegue en internet y demo final.

👉 **Trae tus notas** de las pruebas del prototipo. Las usaremos para decidir qué pulir.